# ClickHouse connection

Connects to the ClickHouse `imdb` database with the official [`clickhouse-connect`](https://clickhouse.com/docs/integrations/python) driver and runs a couple of sample queries.

The data is the [IMDB example dataset](https://clickhouse.com/docs/getting-started/example-datasets/imdb) from the ClickHouse docs. The former Neo4j graph's relationships (`ACTED_IN`, `DIRECTED`, `IN_GENRE`) become the join tables `roles`, `movie_directors` and `genres`:

```
movies(id UInt32, name String, year UInt32, rank Float32)        -- ~388k rows, rank 0 = unrated
actors(id UInt32, first_name String, last_name String, gender FixedString(1))
directors(id UInt32, first_name String, last_name String)
roles(actor_id UInt32, movie_id UInt32, role String, created_at DateTime)   -- ~3.4M rows
movie_directors(director_id UInt32, movie_id UInt64)
genres(movie_id UInt32, genre String)
```

Dataset quirks worth knowing:

- Titles move a leading article to the end: `Matrix, The`, `Few Good Men, A`.
- Movies span 1888-2008; only ~67k of them have a `rank` (IMDb rating) - `rank = 0` means unrated.
- `first_name` may carry a disambiguation suffix (`Chris (I)`); the display name is
  `concat(splitByString(' (', first_name)[1], ' ', last_name)`.
- There is no popularity column. Cast size (`count()` of `roles` per movie) is a good proxy and keeps
  obscure one-actor films with a 9.8 rank out of "top rated" lists.

Create the tables and load the data first with `python clickhouse/load_imdb.py` (the server pulls the dataset from S3).

## 1. Connection settings

Connection details are read from environment variables (optionally via a `.env` file loaded with `python-dotenv`):

| Variable | Default |
|---|---|
| `CLICKHOUSE_HOST` | `localhost` |
| `CLICKHOUSE_PORT` | `8123` (`8443` when `CLICKHOUSE_SECURE=true`) |
| `CLICKHOUSE_USERNAME` | `default` |
| `CLICKHOUSE_PASSWORD` | *(empty)* |
| `CLICKHOUSE_DATABASE` | `imdb` |
| `CLICKHOUSE_SECURE` | `false` |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file if present; never hard-code credentials here

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_SECURE = os.getenv("CLICKHOUSE_SECURE", "false").strip().lower() in ("1", "true", "yes")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT") or (8443 if CLICKHOUSE_SECURE else 8123))
CLICKHOUSE_USERNAME = os.getenv("CLICKHOUSE_USERNAME", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "imdb")

## 2. Install the driver

In [ ]:
%pip install clickhouse-connect python-dotenv pandas

## 3. Create the client and verify connectivity

`get_client()` opens an HTTP(S) client; `ping()` returns `True` when the server answers.

In [ ]:
import clickhouse_connect

# HTTP interface: 8123 (plain) / 8443 (TLS, e.g. ClickHouse Cloud)
client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USERNAME,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
    secure=CLICKHOUSE_SECURE,
)

assert client.ping(), "ClickHouse server is not reachable"
print("Connected to ClickHouse", client.server_version, "| database:", client.database)

## 4. Sample query: first 25 rows

Neo4j's `MATCH (n) RETURN n LIMIT 25` has no single-table equivalent, so we first count rows per table (the analogue of counting nodes/relationships) and then fetch 25 movies.

In [ ]:
for table in ("movies", "actors", "directors", "roles", "movie_directors", "genres"):
    print(f"{table:<16}", client.command(f"SELECT count() FROM {table}"))

In [ ]:
import time

sql = """
    SELECT * FROM movies LIMIT 25
"""
start = time.perf_counter()
result = client.query(sql)
elapsed_ms = (time.perf_counter() - start) * 1000

In [ ]:
# Loop through results and do something with them
for row in result.named_results():
    print(row)  # each row is a dict keyed by column name

# Summary information
print("The query `{query}` returned {rows} rows in {ms:.1f} ms.".format(
    query=" ".join(sql.split()), rows=result.row_count, ms=elapsed_ms,
))

## 5. Parameterised query: top movies in a genre

Equivalent of `MATCH (m:Movie)-[:IN_GENRE]->(g:Genre) WHERE g.name = "Horror" ...`. The genre and minimum cast size are passed with ClickHouse server-side binding (`{genre:String}`, `{min_cast:UInt32}`) instead of string formatting. Unrated movies (`rank = 0`) are excluded and movies need at least `min_cast` credited actors so obscure titles don't dominate.

In [ ]:
sql = """
    SELECT m.name AS title, m.year AS year, round(m.rank, 1) AS rank, c.cast_size AS cast_size
    FROM movies AS m
    INNER JOIN genres AS g ON g.movie_id = m.id
    INNER JOIN (
        SELECT movie_id, count() AS cast_size
        FROM roles
        GROUP BY movie_id
        HAVING cast_size >= {min_cast:UInt32}
    ) AS c ON c.movie_id = m.id
    WHERE g.genre = {genre:String} AND m.rank > 0
    ORDER BY m.rank DESC, c.cast_size DESC
    LIMIT 25
"""
params = {"genre": "Horror", "min_cast": 30}
start = time.perf_counter()
result = client.query(sql, parameters=params)
elapsed_ms = (time.perf_counter() - start) * 1000

In [ ]:
# Loop through results and do something with them
for row in result.named_results():
    print(row)  # each row is a dict keyed by column name

# Summary information
print("The query `{query}` returned {rows} rows in {ms:.1f} ms.".format(
    query=" ".join(sql.split()), rows=result.row_count, ms=elapsed_ms,
))

## 6. Same query as a pandas DataFrame

In [ ]:
df = client.query_df(sql, parameters=params)
df

In [ ]:
client.close()